# Comparing runs with 16 and 32 frames on WLASL 100 to 2000, with similar parameters

In [1]:
from typing import cast
import json
from pathlib import Path
#locals
# from code.run_types import ResSet, RunRes
import pandas as pd
from run_types import ResSet, RunRes, GenInfo
from resulting import print_json,  RESULTS_DIR, load_config_and_find_runs

In [2]:
acc_cuttoff = 0
avail_models = ['MViTv2_S_16x4', 'MViTv2_B_32x3', 'S3D']

additional_modifications = {
    'results': {
        'best_val_acc': lambda x: x > acc_cuttoff
    },
    'admin': {
        'model': lambda x: x in avail_models
    }
}

exclude_keys = [
    ['results', 'test_shuff'],
    ['results', 'check_name'],
]

In [3]:
runs_16_p = RESULTS_DIR / 'config_16f_15p.toml'
runs_32_p = RESULTS_DIR / 'config_32f_15p.toml'
assert runs_16_p.exists(), f"{runs_16_p} not found"
assert runs_32_p.exists(), f"{runs_32_p} not found"

In [4]:
def load_find(conf_path: Path) -> GenInfo:
    runs =  load_config_and_find_runs(
            conf_path,
            exclude=exclude_keys,
            extra_mods=additional_modifications
        )
    assert runs is not None
    return runs

## Parameters in common:

The only parameter that differs is the number of frames. In all cases, models trained on asl300 upward were initialised from the previous split. 

In [5]:
runs_16 = load_find(runs_16_p)
runs_32 = load_find(runs_32_p)

{
    "training": {
        "batch_size_equivalent": 8
    },
    "optimizer": {
        "eps": 1e-05,
        "backbone_init_lr": 0.0001,
        "backbone_weight_decay": 0.001,
        "classifier_init_lr": 0.001,
        "classifier_weight_decay": 0.001
    },
    "model_params": {
        "drop_p": 0.5
    },
    "data": {
        "num_frames": 16,
        "frame_size": 224,
        "train_augs": {
            "frame_size_strategy": "Random_crop",
            "normalise": true,
            "frame_sampler": {
                "method": "og"
            },
            "spatial_aug": [
                {
                    "type": "HORIZONTAL_FLIP",
                    "p": 0.5
                }
            ]
        },
        "test_augs": {
            "frame_size_strategy": "Centre_crop",
            "normalise": true,
            "frame_sampler": {
                "method": "og"
            }
        }
    },
    "early_stopping": {
        "metric": [
            "val",
          

## Runs 16 frames:

Lets check how many runs there are that match that spec:

In [6]:
for run in runs_16['results']:
    print_json(run['admin'])
    print()
print(len(runs_16['results']))

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "runs/asl2000/S3D/exp001/checkpoints",
    "exp_no": "001",
    "recover": false,
    "config_path": "configfiles/asl2000/S3D/exp001.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl1000/S3D/exp001/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl1000",
    "save_path": "runs/asl1000/S3D/exp001/checkpoints",
    "exp_no": "001",
    "recover": false,
    "config_path": "configfiles/asl1000/S3D/exp001.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl300/S3D/exp003/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl300",
    "save_path": "runs/asl300/S3D/exp003/checkpoints",
    "exp_no": "003",
    "recover": false,
    "config_path": "configfiles/asl300/S3D/exp003.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl100/S3D/exp075/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLAS

## Runs 32 frames:

Lets check how many runs there are that match that spec:

<!-- Note there were 3 extra asl100s. We will take exp007 because it is used as a starting point for the asl300 run -->


In [7]:
# runs_32['results'] = runs_32['results'][:-3] 
for run in runs_32['results']:
    print_json(run['admin'])
    print()
print(len(runs_32['results']))

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "runs/asl2000/S3D/exp002/checkpoints",
    "exp_no": "002",
    "recover": false,
    "config_path": "configfiles/asl2000/S3D/exp002.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl1000/S3D/exp002/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl1000",
    "save_path": "runs/asl1000/S3D/exp002/checkpoints",
    "exp_no": "002",
    "recover": false,
    "config_path": "configfiles/asl1000/S3D/exp002.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl300/S3D/exp004/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLASL",
    "split": "asl300",
    "save_path": "runs/asl300/S3D/exp004/checkpoints",
    "exp_no": "004",
    "recover": false,
    "config_path": "configfiles/asl300/S3D/exp004.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl100/S3D/exp076/checkpoints/best.pth"
}

{
    "model": "S3D",
    "dataset": "WLAS

## Now we can compare the runs

In [8]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

#### Need to modify the names of S3D or they all get mixed together

In [9]:
df_format = []
for res in runs_16['results']: # + runs_32['results'] :
    model_name = res['admin']['model']
    if model_name == 'S3D':
        model_name += '_16'
    df_format.append(
        {'model': model_name, 'subset': res['admin']['split']} | {k : v for k, v in res['results'][set_name][acc_type].items()}
    )

for res in runs_32['results'] :
    model_name = res['admin']['model']
    if model_name == 'S3D':
        model_name += '_32'
    df_format.append(
        {'model': model_name, 'subset': res['admin']['split']} | {k : v for k, v in res['results'][set_name][acc_type].items()}
    )


df = pd.DataFrame(df_format)

In [10]:
df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})
# df

In [11]:
df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')

In [12]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']
for set_name in subsets:
    subdf = df[df['subset'] == set_name]
    print(f'{set_name}'.capitalize())
    display(subdf.sort_values('Top-1', ascending=False))

Asl100


,model,subset,Top-1,Top-5,Top-10
23,MViTv2_B_32x3,asl100,78.29,90.31,95.74
17,MViTv2_B_32x3,asl100,77.52,93.41,96.51
18,MViTv2_B_32x3,asl100,74.42,92.25,96.12
7,MViTv2_S_16x4,asl100,73.64,91.47,96.12
24,MViTv2_B_32x3,asl100,73.26,92.64,95.74
21,S3D_32,asl100,64.73,86.82,92.25
22,S3D_32,asl100,63.18,89.53,93.80
19,S3D_32,asl100,60.08,86.43,91.86
20,S3D_32,asl100,59.30,84.11,91.47
26,S3D_32,asl100,59.30,86.43,92.64


Asl300


,model,subset,Top-1,Top-5,Top-10
16,MViTv2_B_32x3,asl300,67.07,88.77,92.22
6,MViTv2_S_16x4,asl300,60.93,88.02,92.51
11,S3D_32,asl300,40.12,64.97,76.50
2,S3D_16,asl300,35.33,64.82,76.05


Asl1000


,model,subset,Top-1,Top-5,Top-10
15,MViTv2_B_32x3,asl1000,56.77,83.48,89.66
5,MViTv2_S_16x4,asl1000,51.12,79.37,86.19
1,S3D_16,asl1000,30.38,59.97,70.58
10,S3D_32,asl1000,28.46,57.94,69.72


Asl2000


,model,subset,Top-1,Top-5,Top-10
14,MViTv2_B_32x3,asl2000,44.56,76.35,83.71
4,MViTv2_S_16x4,asl2000,39.91,70.61,79.65
9,S3D_32,asl2000,26.71,56.10,67.11
0,S3D_16,asl2000,23.38,51.23,64.19
13,MViTv2_B_32x3,asl2000,0.14,0.59,1.01
